In [1]:
from pathlib import Path
import cv2
import dask.array as da
import tifffile as tf
import numpy as np 
from skimage.color import rgb2hed, hed2rgb
from dask.diagnostics import ProgressBar
import skimage
import pickle 
import matplotlib.pyplot as plt
from tqdm import tqdm 

In [2]:
def register(reference_image, image_to_register, algorithm, coord=None, image_to_warp=None, warp_shape=None):
    if(algorithm == 'SIFT'):
        orb = cv2.SIFT_create(contrastThreshold=0.005, edgeThreshold=5, sigma=1.2)
    else:
        orb = cv2.ORB_create()
    
    if(coord is not None):
        (keypoints1, descriptors1) = orb.detectAndCompute(reference_image[coord[0]:coord[1], coord[2]:coord[3]], None)
        (keypoints2, descriptors2) = orb.detectAndCompute(image_to_register[coord[0]:coord[1], coord[2]:coord[3]], None)
    else:
        (keypoints1, descriptors1) = orb.detectAndCompute(reference_image, None)
        (keypoints2, descriptors2) = orb.detectAndCompute(image_to_register, None)

    if(algorithm == 'SIFT'):
        FLANN_INDEX_KDTREE = 1
        index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
        search_params = dict(checks=50)
        flann = cv2.FlannBasedMatcher(index_params, search_params)

        matches = flann.knnMatch(descriptors1, descriptors2, k=2)
        good_matches = []
        for m, n in matches:
            if m.distance < 0.7 * n.distance:
                good_matches.append(m)
        points1 = np.float32([keypoints1[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
        points2 = np.float32([keypoints2[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
    else:
        bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
        matches = bf.match(descriptors1, descriptors2)
        good_matches = sorted(matches, key=lambda x: x.distance)
        points1 = np.zeros((len(good_matches), 2), dtype=np.float32)
        points2 = np.zeros_like(points1)
        for i, match in enumerate(good_matches):
            points1[i, :] = keypoints1[match.queryIdx].pt
            points2[i, :] = keypoints2[match.trainIdx].pt


    """min_good_matches = 4
    if len(good_matches) < min_good_matches:
        raise Exception(f"Not enough good matches found - {len(good_matches)} found, but {min_good_matches} needed.")
    """
    homography, _ = cv2.findHomography(points1, points2, cv2.RANSAC, 10)
    print(homography.shape)
    if(warp_shape is None):
        if(image_to_warp is None):
            height, width = reference_image.shape[:2]
        else:
            height, width = image_to_warp.shape[:2]
    else:
        height, width = warp_shape
    if(image_to_warp is None):
        registered_image = skimage.transform.warp(image_to_register, np.array(homography), output_shape=(height, width)) 
    else:
        registered_image = skimage.transform.warp(image_to_warp, np.array(homography), output_shape=(height, width))

    return homography, registered_image


In [3]:
# === registration method === 
def register_fluorescent_dapi_images(reference_image_key, DAPI_to_register, algorithm, registration_base_path, key, coord=None):
    with ProgressBar():
        reference_image_computed = DAPI_to_register[reference_image_key].compute()

    complete_registration_base_path = registration_base_path + '/' + algorithm + key

    DAPI_registered = {}
    H_for_registration = {}

    reference_image_computed = cv2.convertScaleAbs(reference_image_computed,  alpha=(255.0/65535.0)).astype(np.uint8)
    for cycle_channel, cycle_image in DAPI_to_register.items():
        """if(reference_image_key == cycle_channel):
            DAPI_registered[cycle_channel] = reference_image_computed
            continue"""
        with ProgressBar():    
            cycle_image_computed = cycle_image.compute()
        cycle_image_computed = cv2.convertScaleAbs(cycle_image_computed,  alpha=(255.0/65535.0)).astype(np.uint8)
        H, cycle_image_registered = register(reference_image_computed, cycle_image_computed, algorithm, coord)
        DAPI_registered[cycle_channel] = cycle_image_registered
        del cycle_image_registered
        H_for_registration[cycle_channel] = H
        del H
    
    for cycle_channel, cycle_image in DAPI_registered.items():
        cycle = cycle_channel.split('_')[0]
        channel = cycle_channel.split('_')[-1]
        cycle_key = cycle + '_' + channel
        print(cycle_key)
        tf.imwrite(complete_registration_base_path + '/' + cycle_key + '_registered.tif', cycle_image)

    del DAPI_to_register

    with open(complete_registration_base_path + '/IF_H_for_registration.pickle', 'wb') as f: 
        pickle.dump(H_for_registration, f)



    return DAPI_registered, H_for_registration

def register_fluorescent_non_dapi_images(reference_image_key, non_DAPI_to_register, H_for_registration, algorithm, registration_base_path, key):
    complete_registration_base_path = registration_base_path + '/' + algorithm + key

    non_DAPI_registered = {}

    for cycle_channel, cycle_image in non_DAPI_to_register.items():
        with ProgressBar():    
            cycle_image_computed = cycle_image.compute()
        cycle = cycle_channel.split('_')[0]
        cycle_key = cycle +  '_CH1'
        height, width = cycle_image_computed.shape[:2]
        """if(cycle_key == reference_image_key):
            cycle_image_registered = cycle_image_computed
        else:"""
        cycle_image_registered = skimage.transform.warp(cycle_image_computed, np.array(H_for_registration[cycle_key]), output_shape=(height, width))
        non_DAPI_registered[cycle_channel] = cycle_image_registered
    
    for cycle_channel, cycle_image in non_DAPI_registered.items():
        cycle = cycle_channel.split('_')[0]
        channel = cycle_channel.split('_')[-1]
        cycle_key = cycle + '_' + channel
        print(cycle_key)
        tf.imwrite(complete_registration_base_path + '/' + cycle_key + '_registered.tif', cycle_image)
    
    return non_DAPI_registered

# for each algorithm, register images for a sample 
# for H&E, extract hematoxylin 

def register_color_images(DAPI_for_registration, stain_images_to_register, algorithm, registration_base_path, key, coord=None):
    complete_registration_base_path = registration_base_path + '/' + algorithm + key

    stain_images_registered = {}
    H_for_registration = {}
    color_scalebar_dict = {}
    for stain, stain_image in stain_images_to_register.items(): 
        with ProgressBar():
            stain_image_computed = stain_image.compute()
        if(stain == 'H&E'):
            
            stain_image_resized_computed = cv2.resize(stain_image_computed, (DAPI_for_registration.shape[1], DAPI_for_registration.shape[0]))
            color_scalebar_dict[stain] = 0.5319*(stain_image_computed.shape[1]/stain_image_resized_computed.shape[1])

            
            if(coord is not None):
                stain_hed = skimage.exposure.equalize_adapthist(stain_image_resized_computed.copy()[coord[0]:coord[1], coord[2]:coord[3]])
                stain_hed = rgb2hed(stain_hed)
                null = np.zeros_like(stain_hed[:, :, 0])
                stain_h = hed2rgb(np.stack((stain_hed[:, :, 0], null, null), axis=-1))
                stain_h = cv2.bitwise_not(((stain_h / np.max(stain_h)) * 255).astype(np.uint8))
                stain_h = cv2.cvtColor(stain_h, cv2.COLOR_RGB2GRAY)

                H, stain_registered = register(DAPI_for_registration[coord[0]:coord[1], coord[2]:coord[3]], stain_h, algorithm, image_to_warp=stain_image_resized_computed)
                H_for_registration[stain] = H 
                
            else:
                stain_hed = skimage.exposure.equalize_adapthist(stain_image_resized_computed)
                stain_hed = rgb2hed(stain_hed)
                null = np.zeros_like(stain_hed[:, :, 0])
                stain_h = hed2rgb(np.stack((stain_hed[:, :, 0], null, null), axis=-1))
                stain_h = cv2.bitwise_not(((stain_h / np.max(stain_h)) * 255).astype(np.uint8))
                stain_h = cv2.cvtColor(stain_h, cv2.COLOR_RGB2GRAY)
                                       
                H, stain_registered = register(DAPI_for_registration, stain_h, algorithm, image_to_warp=stain_image_resized_computed)
                H_for_registration[stain] = H


            stain_images_registered[stain] = stain_registered
        
        else:
            reference = (stain_images_registered['H&E'] / np.max(stain_images_registered['H&E']) *255).astype(np.uint8)
            stain_image_resized_computed = cv2.resize(stain_image_computed, (reference.shape[1], reference.shape[0]))

            reference = cv2.cvtColor(reference, cv2.COLOR_RGB2GRAY)
            stain_image_resized_computed_gray = cv2.cvtColor(stain_image_resized_computed, cv2.COLOR_RGB2GRAY)

            H, stain_registered = register(reference.astype(np.uint8), stain_image_resized_computed_gray.astype(np.uint8), algorithm, coord, image_to_warp=stain_image_resized_computed)
            H_for_registration[stain] = H 
            stain_images_registered[stain] = stain_registered
    
    for stain, image in stain_images_registered.items():
        print(stain)
        tf.imwrite(complete_registration_base_path + '/' + stain + '_registered.tif', image)

    with open(complete_registration_base_path + '/color_H_for_registration.pickle', 'wb') as f: 
        pickle.dump(H_for_registration, f)

        
    return stain_images_registered, color_scalebar_dict

In [4]:
imaging_base_path = 'Y:/coskun-lab/Aditi/01/Imaging'

In [5]:
algorithms = ['SIFT', 'ORB']

In [6]:
color_stains = ['H&E', 'PAS', 'Trichrome']

In [7]:
key = '/Multiplex 12/Rqo20'

In [8]:
roi_path = 'Y:/coskun-lab/Aditi/01/Results/registration' + key + '/1x'

In [9]:
registration_base_path =  'Y:/coskun-lab/Aditi/01/Results/registration_algorithms_comparison'

In [10]:
roi_exists = False

In [11]:
if(roi_exists):
    with open(roi_path + '/roi_coord_dict.pickle', 'rb') as f: 
        roi = pickle.load(f)

In [12]:
# === create folder for registered images == 
Path(registration_base_path).mkdir(exist_ok=True)
for algorithm in algorithms: 
    Path(registration_base_path + '/' + algorithm).mkdir(exist_ok=True)
    Path(registration_base_path + '/' + algorithm + '/' + key.split('/')[1]).mkdir(exist_ok=True)
    Path(registration_base_path + '/' + algorithm + key).mkdir(exist_ok=True)

In [13]:
def resize_chunk(chunk, factor):
    return cv2.resize(chunk, None, fx=1/factor, fy=1/factor)

In [14]:
def invert_chunk(chunk):
    return cv2.bitwise_not(chunk)

In [15]:
def white_balance(chunk, white_patch_average, image_average):
    return (chunk * (white_patch_average/image_average)).clip(0, 255).astype((int)).astype(np.uint8)

In [16]:
# === load fluorescent and RGB images === 
def load_fluorescent_dapi_images(factor_downsample):
    fluorescent_images_path = Path(imaging_base_path + '/IF')
    print(fluorescent_images_path)
    dapi_paths = [f for f in fluorescent_images_path.rglob('*.tif') if 'Bleach' not in str(f) and 'stitched_background_removed' in str(f) and 'CH1' in str(f) and key.split('/')[2] in str(f)]
   
    DAPI_to_register_list = [f for f in dapi_paths] 
    DAPI_to_register = {}
    IF_scalebar_dict = {}

    for path in tqdm(DAPI_to_register_list):
        cycle_index = str(path).find('Cycle')
        ch_index = str(path).find('CH')
        cycle = str(path)[cycle_index: cycle_index + len('Cycle')+2]
        channel = str(path)[ch_index: ch_index + len('CH')+1]
        #print(cycle + '_' + channel)
        img = da.from_array(tf.imread(path), chunks=(512, 512))
        
        img = da.pad(img, ((0, int(np.ceil(img.shape[0]/512)*512-img.shape[0])),  (0, int(np.ceil(img.shape[1]/512)*512-img.shape[1]))))
        img = img.rechunk((512, 512), balance=True)
        img = img.map_blocks(resize_chunk, factor_downsample, dtype=np.uint16, chunks=(int(512/factor_downsample), int(512/factor_downsample)))
        img = da.pad(img, ((0, int(np.ceil(img.shape[0]/512)*512-img.shape[0])),  (0, int(np.ceil(img.shape[1]/512)*512-img.shape[1]))))
        img = img.rechunk(chunks=(512, 512), balance=True)
        DAPI_to_register[cycle + '_' + channel] = img
        IF_scalebar_dict[cycle + '_' + channel] = 0.1771*factor_downsample
        del img
    
    return DAPI_to_register, IF_scalebar_dict
    
def load_fluorescent_non_dapi_images(factor_downsample, IF_scalebar_dict):
    
    fluorescent_images_path = Path(imaging_base_path + '/IF')
    non_dapi_paths = [f for f in fluorescent_images_path.rglob('*.tif') if 'Bleach' not in str(f) and 'stitched_background_removed' in str(f) and 'CH1' not in str(f) and 'CH2' not in str(f) and key.split('/')[2] in str(f)]
    non_DAPI_to_register_list = [f for f in non_dapi_paths]
    non_DAPI_to_register = {}

    for path in tqdm(non_DAPI_to_register_list):
        cycle_index = str(path).find('Cycle')
        ch_index = str(path).find('CH')
        cycle = str(path)[cycle_index: cycle_index + len('Cycle')+2]
        channel = str(path)[ch_index: ch_index + len('CH')+1]
        #print(cycle + '_' + channel)
        img = da.from_array(tf.imread(path), chunks=(512, 512))

        img = da.pad(img, ((0, int(np.ceil(img.shape[0]/512)*512-img.shape[0])),  (0, int(np.ceil(img.shape[1]/512)*512-img.shape[1]))))
        img = img.rechunk((512, 512), balance=True)
        img = img.map_blocks(resize_chunk, factor_downsample, dtype=np.uint16, chunks=(int(512/factor_downsample), int(512/factor_downsample)))
        img = da.pad(img, ((0, int(np.ceil(img.shape[0]/512)*512-img.shape[0])),  (0, int(np.ceil(img.shape[1]/512)*512-img.shape[1]))))
        img = img.rechunk(chunks=(512, 512), balance=True)
        IF_scalebar_dict[cycle + '_' + channel] = 0.1771*factor_downsample
        non_DAPI_to_register[cycle + '_' + channel] = img
        del img 
    
    return non_DAPI_to_register, IF_scalebar_dict

def load_color_images():
    stain_images_to_register = {}
    for stain in color_stains:
        print(stain)
        if(stain == 'PAS'):
            stain_path = Path(imaging_base_path + '/' + stain + '_'.join(key.split(' ')))
        else:
            stain_path = Path(imaging_base_path + '/' + stain + key) 
        stain_img_path = [f for f in stain_path.rglob('*.tif') if 'Raw.ome' in str(f)]
        stain_img = da.rot90(da.from_array(tf.imread(stain_img_path[0]), chunks=(512, 512, 3)), k=1)
        stain_img = da.pad(stain_img, ((0, int(np.ceil(stain_img.shape[0]/512)*512-stain_img.shape[0])),  (0, int(np.ceil(stain_img.shape[1]/512)*512-stain_img.shape[1])), (0, 0)))
        stain_img = stain_img.rechunk(chunks=(512, 512, 3), balance=True)

        white_patch = stain_img[:1000, :1000].compute()
        white_patch_average = white_patch[0].mean()
        stain_img_average = da.mean(stain_img).compute()
        stain_img = stain_img.map_blocks(white_balance, white_patch_average, stain_img_average)

        stain_images_to_register[stain] = stain_img

    return stain_images_to_register  


In [17]:
def load_bleached_fluorescent_images(factor_downsample):
    fluorescent_images_path = Path(imaging_base_path + '/IF')

    dapi_paths = [f for f in fluorescent_images_path.rglob('*.tif') if 'Bleach' in str(f) and 'stitched_background_removed' not in str(f) and 'CH1' in str(f) and key.split('/')[2] in str(f)]
    
    DAPI_to_register_list = [f for f in dapi_paths] 
    DAPI_to_register = {}

    for path in tqdm(DAPI_to_register_list):
        cycle_index = str(path).find('Cycle')
        ch_index = str(path).find('CH')
        region_index = str(path).find('Region')
        cycle = str(path)[cycle_index: cycle_index + len('Cycle')+2]
        channel = str(path)[ch_index: ch_index + len('CH')+1]
        region = str(path)[region_index: region_index + len('Region')+2]
        #print(cycle + '_' + channel)
        img = resize_chunk(tf.imread(path), factor_downsample)
        
        """img = da.pad(img, ((0, int(np.ceil(img.shape[0]/512)*512-img.shape[0])),  (0, int(np.ceil(img.shape[1]/512)*512-img.shape[1]))))
        img = img.rechunk((512, 512), balance=True)
        img = img.map_blocks(resize_chunk, factor_downsample, dtype=np.uint16, chunks=(int(512/factor_downsample), int(512/factor_downsample)))
        img = da.pad(img, ((0, int(np.ceil(img.shape[0]/512)*512-img.shape[0])),  (0, int(np.ceil(img.shape[1]/512)*512-img.shape[1]))))
        img = img.rechunk(chunks=(512, 512), balance=True)"""

        if(DAPI_to_register.get(cycle + '_' + channel) !=None):
            DAPI_to_register[cycle + '_' + channel][region] = img
        else:
            DAPI_to_register[cycle + '_' + channel] = {}
            DAPI_to_register[cycle + '_' + channel][region] = img
        del img
    
    non_dapi_paths = [f for f in fluorescent_images_path.rglob('*.tif') if 'Bleach' in str(f) and 'stitched_background_removed' not in str(f) and 'stitched' in str(f) and 'CH1' not in str(f) and key.split('/')[2] in str(f)]
    non_DAPI_to_register_list = [f for f in non_dapi_paths]
    non_DAPI_to_register = {}

    for path in tqdm(non_DAPI_to_register_list):
        cycle_index = str(path).find('Cycle')
        ch_index = str(path).find('CH')
        region_index = str(path).find('Region')
        cycle = str(path)[cycle_index: cycle_index + len('Cycle')+2]
        channel = str(path)[ch_index: ch_index + len('CH')+1]
        region = str(path)[region_index: region_index + len('Region')+2]
        #print(cycle + '_' + channel)
        img = resize_chunk(tf.imread(path), factor_downsample)

        """img = da.pad(img, ((0, int(np.ceil(img.shape[0]/512)*512-img.shape[0])),  (0, int(np.ceil(img.shape[1]/512)*512-img.shape[1]))))
        img = img.rechunk((512, 512), balance=True)
        img = img.map_blocks(resize_chunk, factor_downsample, dtype=np.uint16, chunks=(int(512/factor_downsample), int(512/factor_downsample)))
        img = da.pad(img, ((0, int(np.ceil(img.shape[0]/512)*512-img.shape[0])),  (0, int(np.ceil(img.shape[1]/512)*512-img.shape[1]))))
        img = img.rechunk(chunks=(512, 512), balance=True)"""
        if(non_DAPI_to_register.get(cycle + '_' + channel) !=None):
            non_DAPI_to_register[cycle + '_' + channel][region] = img
        else:
            non_DAPI_to_register[cycle + '_' + channel] = {}
            non_DAPI_to_register[cycle + '_' + channel][region] = img
        del img 
    
    return DAPI_to_register, non_DAPI_to_register


In [ ]:
DAPI_to_register, IF_scalebar_dict = load_fluorescent_dapi_images(4)

In [ ]:
DAPI_to_register

In [ ]:
max_size = [28160, 32768] 
"""[0, 0]"""
"""for cycle_channel_key, img in DAPI_to_register.items():
    if(img.shape[0] > max_size[0]):
        max_size[0] = img.shape[0]
    if(img.shape[1] > max_size[1]):
        max_size[1] = img.shape[1]"""
for cycle_channel_key, img in DAPI_to_register.items():
    DAPI_to_register[cycle_channel_key] = da.pad(img, ((0, max_size[0]-img.shape[0]),  (0, max_size[1]-img.shape[1])))


In [ ]:
DAPI_to_register

In [ ]:
DAPI_registered_SIFT, H_SIFT = register_fluorescent_dapi_images('Cycle 1_CH1', DAPI_to_register, 'SIFT', registration_base_path, key, coord=(0, 25000, 0, 25000))                                            

In [ ]:

DAPI_registered_SIFT = {}

complete_registration_path = registration_base_path + '/SIFT' + key
registered_dapi_image_paths = [f for f in Path(complete_registration_path).rglob('*.tif') if 'CH1' in str(f) and 'bleached' not in str(f)]

for path in tqdm(registered_dapi_image_paths):
    cycle_channel_key = '_'.join(str(path).split('\\')[-1].split('_')[:2])
    DAPI_registered_SIFT[cycle_channel_key] = tf.imread(path)

"""non_DAPI_registered_SIFT = {}
registered_non_dapi_image_paths = [f for f in Path(complete_registration_path).rglob('*.tif') if 'CH1' not in str(f) and 'bleached' not in str(f) and 'Cycle' in str(f)]

for path in tqdm(registered_non_dapi_image_paths):
    cycle_channel_key = '_'.join(str(path).split('\\')[-1].split('_')[:2])
    non_DAPI_registered_SIFT[cycle_channel_key] = tf.imread(path)"""

  0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
DAPI_registered_ORB, H_ORB = register_fluorescent_dapi_images('Cycle 3_CH1', DAPI_to_register, 'ORB', registration_base_path, key)

In [ ]:
DAPI_registered_ORB = {}
non_DAPI_registered_ORB = {}

complete_registration_path = registration_base_path + '/ORB' + key
registered_dapi_image_paths = [f for f in Path(complete_registration_path).rglob('*.tif') if 'CH1' in str(f)]
registered_non_dapi_image_paths = [f for f in Path(complete_registration_path).rglob('*.tif') if 'CH1' not in str(f)]

for path in tqdm(registered_dapi_image_paths):
    cycle_channel_key = '_'.join(str(path).split('\\')[-1].split('_')[:1])
    DAPI_registered_ORB[cycle_channel_key] = tf.imread(path)

for path in tqdm(registered_non_dapi_image_paths):
    cycle_channel_key = '_'.join(str(path).split('\\')[-1].split('_')[:1])
    non_DAPI_registered_ORB[cycle_channel_key] = tf.imread(path)

In [ ]:
"""complete_registration_path = registration_base_path + '/' + 'SIFT' + key
with open(complete_registration_path + '/IF_scalebar_dict.pickle', 'wb') as f: 
    pickle.dump(IF_scalebar_dict, f)"""

In [ ]:
complete_registration_path = registration_base_path + '/' + 'SIFT' + key
with open(complete_registration_path + '/IF_scalebar_dict.pickle', 'rb') as f: 
    IF_scalebar_dict = pickle.load(f)

In [ ]:
non_DAPI_to_register, IF_scalebar_dict = load_fluorescent_non_dapi_images(4, IF_scalebar_dict)

In [ ]:
complete_registration_path = registration_base_path + '/' + 'SIFT' + key
with open(complete_registration_path + '/IF_H_for_registration.pickle', 'rb') as f: 
    H_SIFT = pickle.load(f)

In [ ]:
max_size = [0, 0]
for cycle_channel_key, img in non_DAPI_to_register.items():
    if(img.shape[0] > max_size[0]):
        max_size[0] = img.shape[0]
    if(img.shape[1] > max_size[1]):
        max_size[1] = img.shape[1]
for cycle_channel_key, img in non_DAPI_to_register.items():
    non_DAPI_to_register[cycle_channel_key] = da.pad(img, ((0, max_size[0]-img.shape[0]),  (0, max_size[1]-img.shape[1])))


In [ ]:
non_DAPI_to_register

In [ ]:
non_DAPI_registered_SIFT = register_fluorescent_non_dapi_images('Cycle 1_CH1', non_DAPI_to_register, H_SIFT, 'SIFT', registration_base_path, key)

In [ ]:
fig, ax = plt.subplots(1)
ax.imshow(DAPI_registered_SIFT['Cycle 1_CH1'][12000:13000, 12000:13000], alpha=0.5, cmap='Reds')
ax.imshow(DAPI_registered_SIFT['Cycle 3_CH1'][12000:13000, 12000:13000], alpha=0.5, cmap='Greens')

In [ ]:
for algorithm in algorithms: 
    complete_registration_path = registration_base_path + '/' + algorithm + key
    with open(complete_registration_path + '/IF_scalebar_dict.pickle', 'wb') as f: 
        pickle.dump(IF_scalebar_dict, f)

In [ ]:
non_DAPI_registered_ORB = register_fluorescent_non_dapi_images('Cycle 3_CH1', non_DAPI_to_register, H_ORB, 'ORB', registration_base_path, key)

In [ ]:
color_images = load_color_images()

In [ ]:
reference = DAPI_registered_SIFT['Cycle 1_CH1']

In [ ]:
reference = (reference * 255).astype(np.uint8)

In [ ]:
color_images_registered_SIFT, color_scalebar_dict = register_color_images(reference, color_images, 'SIFT', registration_base_path, key, coord=(0, 15000, 0, 15000))

In [ ]:
fig, ax = plt.subplots(1, 3)
fig.set_figheight(10)
fig.set_figwidth(30)
ax[0].imshow(color_images_registered_SIFT['H&E'][12000:13000, 14000:15000])
ax[1].imshow(color_images_registered_SIFT['PAS'][12000:13000, 14000:15000])
ax[2].imshow(color_images_registered_SIFT['Trichrome'][12000:13000, 14000:15000])

ax[0].axis('off')
ax[1].axis('off')
ax[2].axis('off')

In [ ]:
roi = {
    'A': (16500, 17500, 16500, 17500), 
    'B': (18000, 19000, 14000, 15000), 
    'C': (21000, 22000, 7500, 8500), 
    'D': (12000, 13000, 14000, 15000)
}

In [ ]:
with open(complete_registration_path + '/roi_coord_dict.pickle', 'wb') as f: 
    pickle.dump(roi, f)

In [ ]:
for algorithm in algorithms: 
    complete_registration_path = registration_base_path + '/' + algorithm + key
    with open(complete_registration_path + '/color_scalebar_dict.pickle', 'wb') as f: 
        pickle.dump(color_scalebar_dict, f)

In [ ]:
color_images_registered_ORB = register_color_images(DAPI_to_register['Cycle 3_CH1'], color_images, 'ORB', registration_base_path, key, coord=(0, 15000, 0, 15000))